In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append(str(Path.cwd().parent))
from src.DataLoad_Geo import (
    load_transactions_geo,
    load_geocode_cache,
    load_mrt_stations
)
from src.GeoAnalysis_Aggregations import (
    add_geo_bands,
    summary_by_mrt_band,
    summary_by_city_band,
    summary_by_subzone,
    summary_by_nearest_mrt,
    mrt_band_by_flat_type,
    joint_mrt_city_bands,
    geo_coverage
)
from src.GeoPlot_Aggregations import (
    _save,
    plot_pps_by_mrt_band,
    plot_price_by_mrt_band,
    plot_to_mrt_vs_pps_scatter,
    plot_top_stations_by_volume,
    plot_top_stations_by_pps,
    plot_pps_by_city_band,
    plot_to_city_vs_pps_scatter,
    plot_top_subzones_by_pps,
    plot_bottom_subzones_by_pps,
    plot_joint_mrt_city_heatmap,
    plot_boxplot_pps_by_mrt_band
)

sns.set_theme(style = 'whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# load all datasets
df = load_transactions_geo()
print('shape:', df.shape)
print(geo_coverage(df))
df.head()

Saved cleaned final file -> E:\AI study\HDB_resale_market_analysis\src\..\data\aid\resale_trx_geo.csv
shape: (239265, 26)
{'n': 239265, 'pct_geocoded': np.float64(100.0), 'pct_zone_id': np.float64(100.0), 'pct_to_mrt': np.float64(100.0), 'pct_to_city': np.float64(100.0), 'median_to_mrt': 494.9138692204468, 'median_to_city': 13358.26277718004}


,trx_date,trx_year,trx_month,lease_commence,flat_year,remaining_lease_month,flat_type,flat_model,st,blk,...,lon,postal,subzone,zone_id,nearest_mrt,mrt_lat,mrt_lon,to_mrt,to_city,status
0,2017-01-01,2017,1,1979,38,736,2 ROOM,Improved,ANG MO KIO AVE 10,406,...,103.85,560406,ANG MO KIO,113,ANG MO KIO MRT STATION,1.37,103.85,938.11,8677.74,ok
1,2017-01-01,2017,1,1978,39,727,3 ROOM,New Generation,ANG MO KIO AVE 4,108,...,103.84,560108,ANG MO KIO,113,MAYFLOWER MRT STATION,1.37,103.84,160.05,9782.56,ok
2,2017-01-01,2017,1,1980,37,749,3 ROOM,New Generation,ANG MO KIO AVE 5,602,...,103.84,560602,ANG MO KIO,113,LENTOR MRT STATION,1.38,103.84,391.56,10902.03,ok
3,2017-01-01,2017,1,1980,37,745,3 ROOM,New Generation,ANG MO KIO AVE 10,465,...,103.86,560465,ANG MO KIO,113,ANG MO KIO MRT STATION,1.37,103.85,885.08,9162.28,ok
4,2017-01-01,2017,1,1980,37,749,3 ROOM,New Generation,ANG MO KIO AVE 5,601,...,103.84,NIL,ANG MO KIO,113,LENTOR MRT STATION,1.38,103.84,378.02,10942.85,ok


In [3]:
# geo bands preview
df_b = add_geo_bands(df)
print(df_b['mrt_band'].value_counts(dropna = False).sort_index())
print()
print(df_b['city_band'].value_counts(dropna = False).sort_index())
df_b[['to_mrt', 'to_city', 'mrt_band', 'city_band', 'subzone', 'nearest_mrt', 'price_per_sqm']].head()

mrt_band
0-299m       64196
300-499m     56899
500-799m     64927
800-1199m    37796
1.2-2km      15125
2km+           322
Name: count, dtype: int64

city_band
0-5km       20452
5-10km      44722
10-15km    102447
15-20km     71541
20km+         103
Name: count, dtype: int64


,to_mrt,to_city,mrt_band,city_band,subzone,nearest_mrt,price_per_sqm
0,938.11,8677.74,800-1199m,5-10km,ANG MO KIO,ANG MO KIO MRT STATION,5272.73
1,160.05,9782.56,0-299m,5-10km,ANG MO KIO,MAYFLOWER MRT STATION,3731.34
2,391.56,10902.03,300-499m,10-15km,ANG MO KIO,LENTOR MRT STATION,3910.45
3,885.08,9162.28,800-1199m,5-10km,ANG MO KIO,ANG MO KIO MRT STATION,3897.06
4,378.02,10942.85,300-499m,10-15km,ANG MO KIO,LENTOR MRT STATION,3955.22


In [4]:
# overview of distance to MRT
mrt_band = summary_by_mrt_band(df)
display(mrt_band)
stations = summary_by_nearest_mrt(df, min_n = 30)
display(stations.head(15))
mrt_4rm = mrt_band_by_flat_type(df, flat_type = '4 ROOM')
display(mrt_4rm)

,mrt_band,transactions,mean_price,median_price,mean_pps,median_pps,avg_to_mrt
0,0-299m,64196,563799.34,530000.00,5917.64,5597.01,192.04
1,300-499m,56899,551050.26,518000.00,5751.33,5426.36,394.83
2,500-799m,64927,517848.17,485000.00,5455.78,5214.29,640.13
3,800-1199m,37796,494748.00,465000.00,5195.75,4957.98,967.52
4,1.2-2km,15125,518618.93,500000.00,5166.85,4955.75,1459.05
5,2km+,322,557071.76,542500.00,4976.30,4840.90,2298.58


,nearest_mrt,transactions,mean_pps,median_pps,mean_price,avg_to_mrt
146,YISHUN MRT STATION,10171,4907.93,4807.69,439680.93,889.10
0,ADMIRALTY MRT STATION,7050,4683.01,4702.97,489947.92,571.60
22,BUKIT BATOK MRT STATION,6973,5612.53,5480.77,530480.04,750.98
127,TAMPINES MRT STATION,6767,6115.37,5903.51,606915.03,684.86
2,ANG MO KIO MRT STATION,6510,5798.50,5454.93,508059.63,649.00
75,LAKESIDE MRT STATION,6233,4783.01,4687.50,457269.43,912.96
59,HOUGANG MRT STATION,6023,5217.34,5070.42,514819.67,745.51
126,TAMPINES EAST MRT STATION,5871,4967.03,4814.81,549771.89,605.27
98,PASIR RIS MRT STATION,5811,5121.95,5000.00,599547.66,970.44
70,KHATIB MRT STATION,5408,5222.15,5281.25,498423.49,803.92


,mrt_band,transactions,mean_pps,median_pps,mean_price
0,0-299m,28842,6099.91,5686.57,570958.79
1,300-499m,23167,5894.67,5494.51,554641.38
2,500-799m,26882,5596.52,5294.12,529214.44
3,800-1199m,15945,5205.93,4951.46,500444.72
4,1.2-2km,6712,5162.98,5000.00,491495.44
5,2km+,101,4731.90,4480.00,476929.74


In [5]:
# plot charts about distance to MRT
plot_pps_by_mrt_band(mrt_band)
plot_price_by_mrt_band(mrt_band)
plot_boxplot_pps_by_mrt_band(df)
plot_to_mrt_vs_pps_scatter(df, sample_n = 8000)
plot_top_stations_by_volume(stations, top_n = 15)
plot_top_stations_by_pps(stations, top_n = 15)

Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\pps_by_mrt_band.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\price_by_mrt_band.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\boxplot_pps_by_mrt_band.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\scatter_to_mrt_vs_pps.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\top_stations_by_volume.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\top_stations_by_pps.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\top_stations_by_pps.png'

In [6]:
# overview of distance to city centre
city_band = summary_by_city_band(df)
display(city_band)
plot_pps_by_city_band(city_band)
plot_to_city_vs_pps_scatter(df, sample_n = 8000)

,city_band,transactions,mean_price,median_price,mean_pps,median_pps,avg_to_city
0,0-5km,20452,650857.67,650000.00,7408.72,7000.00,3664.54
1,5-10km,44722,556793.89,500000.00,6159.17,5760.00,7780.90
2,10-15km,102447,535375.14,515000.00,5479.41,5302.01,12937.69
3,15-20km,71541,486419.50,475000.00,4875.81,4782.61,17128.69
4,20km+,103,333891.26,335000.00,3377.17,3310.81,20102.97


Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\pps_by_city_band.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\scatter_to_city_vs_pps.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\scatter_to_city_vs_pps.png'

In [7]:
# overview of subzone/zone_id
zones = summary_by_subzone(df, min_n = 30)
display(zones.head(15))
display(zones.sort_values('median_pps').head(10))
plot_top_subzones_by_pps(zones, top_n = 15)
plot_bottom_subzones_by_pps(zones, top_n = 15)

,zone_id,subzone,transactions,mean_price,median_price,mean_pps,median_pps,avg_to_mrt,avg_to_city
18,142,OUTRAM,1132,811270.54,850500.00,9408.09,9166.67,266.36,1261.81
10,126,DOWNTOWN CORE,52,681228.62,682500.00,8381.50,8323.17,247.69,1435.48
21,147,QUEENSTOWN,6492,652971.11,680000.00,7757.15,7554.95,453.16,6270.35
4,118,BUKIT MERAH,9111,646236.99,661888.00,7337.27,6982.76,527.56,3418.80
28,160,TANGLIN,37,713240.22,620000.00,6875.53,6842.11,60.37,6094.45
6,120,BUKIT TIMAH,651,732539.39,700000.00,6970.81,6704.92,320.27,9075.77
15,131,KALLANG,5362,624017.17,600000.00,7135.35,6544.34,301.97,4020.94
22,149,ROCHOR,681,519190.90,492000.00,6524.68,6441.44,190.68,2387.08
2,115,BISHAN,4162,718672.06,700000.00,6667.32,6422.12,598.24,7736.67
16,136,MARINE PARADE,1448,563885.34,493500.00,6445.85,6339.96,221.30,7145.55


,zone_id,subzone,transactions,mean_price,median_price,mean_pps,median_pps,avg_to_mrt,avg_to_city
14,130,JURONG WEST,15684,468154.92,462000.00,4672.59,4590.91,813.39,17261.36
7,122,CHANGI,51,322071.98,310000.00,4688.83,4621.21,3511.32,19065.45
30,166,WOODLANDS,17049,487919.24,470000.00,4667.78,4628.10,561.94,18449.44
8,124,CHOA CHU KANG,10741,496142.39,495000.00,4640.57,4709.09,508.92,16321.65
19,143,PASIR RIS,6877,597576.75,580000.00,5032.68,4904.76,991.04,14746.46
31,167,YISHUN,16248,456927.03,435000.00,4987.08,4910.71,859.49,15879.05
13,129,JURONG EAST,4841,478300.49,450000.00,4949.27,4921.88,824.66,13820.02
5,119,BUKIT PANJANG,8498,508809.76,488000.00,5018.75,4924.25,273.26,14481.47
1,114,BEDOK,12481,490117.70,435000.00,5427.30,5163.04,566.54,9955.08
12,128,HOUGANG,12061,528766.52,500000.00,5358.49,5242.72,786.74,10629.60


Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\top_subzones_by_pps.png
Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\bottom_subzones_by_pps.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\bottom_subzones_by_pps.png'

In [8]:
# overview of MRT band and city band
joint = joint_mrt_city_bands(df, min_n = 20)
display(joint.head(20))
plot_joint_mrt_city_heatmap(joint, value_col = 'mean_pps')

,city_band,mrt_band,transactions,mean_pps,median_pps
0,0-5km,0-299m,7574,7892.00,7646.26
1,0-5km,300-499m,5440,7557.58,7058.82
2,0-5km,500-799m,5217,7047.82,6340.00
3,0-5km,800-1199m,2082,6290.04,5847.46
4,0-5km,1.2-2km,139,5551.60,5508.47
5,5-10km,0-299m,10264,6606.16,6148.90
6,5-10km,300-499m,12803,6459.55,6028.37
7,5-10km,500-799m,13664,5899.66,5576.92
8,5-10km,800-1199m,6584,5657.72,5298.51
9,5-10km,1.2-2km,1407,5031.98,4932.04


Saved: E:\AI study\HDB_resale_market_analysis\src\..\figures\heatmap_mrt_city_pps.png


'E:\\AI study\\HDB_resale_market_analysis\\src\\..\\figures\\heatmap_mrt_city_pps.png'

In [9]:
# extreme distances
near_mrt = df.loc[df['to_mrt'] < 500, 'price_per_sqm']
far_mrt = df.loc[df['to_mrt'] > 1500, 'price_per_sqm']
near_city = df.loc[df['to_city'] < 8000, 'price_per_sqm']
far_city = df.loc[df['to_city'] > 18000, 'price_per_sqm']
summary = pd.DataFrame({
    'segment': ['MRT < 500m', 'MRT > 1500m', 'City < 8km', 'City > 18km'],
    'n': [near_mrt.shape[0], far_mrt.shape[0], near_city.shape[0], far_city.shape[0]],
    'mean_pps': [near_mrt.mean(), far_mrt.mean(), near_city.mean(), far_city.mean()],
    'median_pps': [near_mrt.median(), far_mrt.median(), near_city.median(), far_city.median()]
})
display(summary)

,segment,n,mean_pps,median_pps
0,MRT < 500m,121095,5839.50,5520.00
1,MRT > 1500m,5910,5108.79,4901.48
2,City < 8km,43456,6937.59,6375.00
3,City > 18km,25535,4740.78,4626.25


In [10]:
# limitations
### Geo feature notes
'''
1. **to_mrt** = metres to nearest MRT/LRT **exit** (from LTA exit GeoJSON), not station centroid (unless you built the station-mode file).
2. **to_city** = metres to Raffles Place `(1.2840, 103.8515)`.
3. **zone_id / subzone** = OneMap planning area (point-in-polygon + name→id map), cached offline.
4. Rows with failed geocode have null `lat` / distances; charts use non-null subsets.
5. Band summaries are descriptive; they do not control for flat type, size, or lease unless filtered (e.g. 4 ROOM table above).
'''

'\n1. **to_mrt** = metres to nearest MRT/LRT **exit** (from LTA exit GeoJSON), not station centroid (unless you built the station-mode file).\n2. **to_city** = metres to Raffles Place `(1.2840, 103.8515)`.\n3. **zone_id / subzone** = OneMap planning area (point-in-polygon + name→id map), cached offline.\n4. Rows with failed geocode have null `lat` / distances; charts use non-null subsets.\n5. Band summaries are descriptive; they do not control for flat type, size, or lease unless filtered (e.g. 4 ROOM table above).\n'